# 03 - From topics to business answers

**Business questions:** when customers push back on price, what is driving it? And which topics show up on the calls
behind deals we won versus deals we lost?

| Section | What it shows |
|---|---|
| 1 | A three-step cascade for price sentiment: keyword prefilter, `AI_SENTIMENT` with aspects, `AI_CLASSIFY` on negatives only |
| 2 | Topics on won and lost deals |
| 3 | `ANALYTICS.CALL_INSIGHTS_SV`: a semantic view, so analysts (and Cortex Analyst) ask in business terms |
| 4 | Both questions answered through the semantic view |

Runs standalone after `02`. Every section ends in a `CHECK`.

In [ ]:
NOTEBOOK_TAG = "transcript-churn-03"

# Session: Snowsight Workspaces supplies one; a local kernel (Cortex Code Desktop,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import time

import numpy as np
import pandas as pd

DB = "TRANSCRIPT_CHURN_ML_DEMO"
WH = "TRANSCRIPT_CHURN_ML_WH"
ROLE = "SYSADMIN"
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"
LLM = "claude-sonnet-4-5"
# The account this demo is allowed to build in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()      # collect(): CALL and EXECUTE cannot be described first
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def one(text):
    return session.sql(text).collect()[0][0]


account, role = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_ROLE()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this demo may only build in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT in this cell.")
if not session.sql(f"SHOW SCHEMAS LIKE 'RAW' IN DATABASE {DB}").collect():
    raise SystemExit(f"STOP: {DB}.RAW is missing. Run lab/setup.sql (./demo.sh setup) first.")
print(f"Connected as role {role} on {account}, warehouse {WH}, query tag {NOTEBOOK_TAG} "
      f"({'local kernel' if LOCAL else 'Workspaces session'})")

In [ ]:
n_topics = one(f"SELECT COUNT(*) FROM {DB}.ML.SEGMENT_TOPICS")
if n_topics == 0:
    raise SystemExit("STOP: ML.SEGMENT_TOPICS is empty. Run notebooks 01 and 02 first.")
ASPECTS = ["pricing model", "competitor price", "value for money"]
session.sql(f"DROP TABLE IF EXISTS {DB}.ANALYTICS.PRICE_SENTIMENT_PROTO").collect()
print(f"{n_topics:,} topic-tagged segments; price aspects {ASPECTS}")

---
## 1. Price sentiment, cheapest step first

Running an LLM over every segment to find price complaints pays for thousands of calls that never mention price. The
cascade spends in proportion to signal:

1. **Prefilter (free):** segments tagged `PRICING`, or whose customer turns mention price, cost, seats, quotes or the bill.
2. **`AI_SENTIMENT`** on the customer's words only, with three aspects: `pricing model`, `competitor price`,
   `value for money` (up to 10 aspects of 30 characters or fewer). The overall label decides whether the segment is negative.
3. **`AI_CLASSIFY`** only on the negatives, to name the single main driver.

In [ ]:
%%sql -r price_sentiment
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.PRICE_SENTIMENT
  COMMENT = 'Price sentiment per prefiltered segment: AI_SENTIMENT aspects, then AI_CLASSIFY driver on negatives' AS
WITH pre AS (
  SELECT t.SEGMENT_ID, t.ACCOUNT_ID, t.CALL_TS, t.TOPIC_ID, e.REDACTED_CUSTOMER_TEXT AS CUSTOMER_TEXT
  FROM TRANSCRIPT_CHURN_ML_DEMO.ML.SEGMENT_TOPICS t
  JOIN TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENT_EMBEDDINGS e USING (SEGMENT_ID)
  WHERE t.TOPIC_ID = 'PRICING'
     OR REGEXP_LIKE(e.REDACTED_CUSTOMER_TEXT, '.*(price|cost|pay|expensive|cheaper|quote|seat|bill).*', 'is')),
s AS (SELECT *, AI_SENTIMENT(CUSTOMER_TEXT, ['pricing model', 'competitor price', 'value for money']) AS SENTIMENT FROM pre),
labeled AS (
  SELECT *, SENTIMENT:categories[0]:sentiment::STRING AS OVERALL_SENTIMENT FROM s)
SELECT SEGMENT_ID, ACCOUNT_ID, CALL_TS, TOPIC_ID, CUSTOMER_TEXT, SENTIMENT, OVERALL_SENTIMENT,
       OVERALL_SENTIMENT IN ('negative', 'mixed') AS IS_NEGATIVE,
       IFF(OVERALL_SENTIMENT IN ('negative', 'mixed'),
           AI_CLASSIFY(CUSTOMER_TEXT, ['competitor price', 'pricing model', 'value for money']):labels[0]::STRING,
           NULL) AS PRICE_DRIVER
FROM labeled

In [ ]:
%%sql -r price_negatives_by_driver
SELECT PRICE_DRIVER, COUNT(*) AS NEGATIVE_SEGMENTS, COUNT(DISTINCT ACCOUNT_ID) AS ACCOUNTS
FROM TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.PRICE_SENTIMENT WHERE IS_NEGATIVE GROUP BY 1 ORDER BY 2 DESC

In [ ]:
total = one(f"SELECT COUNT(*) FROM {DB}.ML.SEGMENT_TOPICS")
g = q(f"""
WITH truth AS (SELECT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4) AS SEGMENT_ID, ANY_VALUE(PRICE_DRIVER) AS PLANTED,
                      ANY_VALUE(TRUE_TOPIC) AS TT FROM {DB}.RAW.SEED_TRUTH GROUP BY 1)
SELECT COUNT(*) AS SENT_CALLS, COUNT_IF(p.IS_NEGATIVE) AS CLASSIFY_CALLS,
       COUNT_IF(p.IS_NEGATIVE AND t.TT = 'PRICING' AND t.PLANTED <> 'positive') AS TRUE_NEG,
       COUNT_IF(p.IS_NEGATIVE AND t.PLANTED = p.PRICE_DRIVER) AS DRIVER_MATCH,
       (SELECT COUNT(DISTINCT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4)) FROM {DB}.RAW.SEED_TRUTH
        WHERE TRUE_TOPIC = 'PRICING' AND PRICE_DRIVER <> 'positive') AS PLANTED_NEG
FROM {DB}.ANALYTICS.PRICE_SENTIMENT p JOIN truth t USING (SEGMENT_ID)""").iloc[0]
PREFILTER_SHARE = int(g.SENT_CALLS) / total
DRIVER_ACC = int(g.DRIVER_MATCH) / int(g.CLASSIFY_CALLS)
RECALL = int(g.TRUE_NEG) / int(g.PLANTED_NEG)
drivers = price_negatives_by_driver.set_index("PRICE_DRIVER").NEGATIVE_SEGMENTS.to_dict()
check("S1", len(drivers) == 3 and min(drivers.values()) > 0 and DRIVER_ACC >= 0.85 and RECALL >= 0.8,
      f"AI_SENTIMENT on {int(g.SENT_CALLS):,} of {total:,} segments ({PREFILTER_SHARE:.0%}), AI_CLASSIFY on "
      f"{int(g.CLASSIFY_CALLS)} negatives; negatives by driver {drivers}; driver matches the planted one "
      f"{DRIVER_ACC:.1%}; recall of planted price complaints {RECALL:.1%}")

---
## 2. Topics on won and lost deals

Segments carry the opportunity of the call, so topics line up with outcomes from the CRM without leaving SQL.

In [ ]:
%%sql -r topics_won_lost
SELECT o.OUTCOME, t.TOPIC_NAME, COUNT(*) AS SEGMENTS,
       ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (PARTITION BY o.OUTCOME), 1) AS PCT_OF_OUTCOME
FROM TRANSCRIPT_CHURN_ML_DEMO.ML.SEGMENT_TOPICS t
JOIN TRANSCRIPT_CHURN_ML_DEMO.RAW.CRM_OPPORTUNITIES o USING (OPPORTUNITY_ID)
GROUP BY 1, 2 ORDER BY 1, 3 DESC

In [ ]:
w = topics_won_lost.pivot_table(index="TOPIC_NAME", columns="OUTCOME", values="PCT_OF_OUTCOME", fill_value=0)
PRICE_LOST, PRICE_WON = float(w.loc["Price and cost", "LOST"]), float(w.loc["Price and cost", "WON"])
check("S2", {"WON", "LOST"} <= set(w.columns) and PRICE_LOST > PRICE_WON,
      f"price and cost is {PRICE_LOST:.1f}% of segments on lost deals vs {PRICE_WON:.1f}% on won deals; "
      f"{len(w)} topics compared")

---
## 3. A semantic view over calls, topics, price sentiment and deals

`ANALYTICS.CALL_INSIGHTS_SV` names the tables, how they join, and the business dimensions and metrics. Analysts query it with
`SEMANTIC_VIEW(...)` in SQL; Cortex Analyst and agents use the same definitions to answer questions in plain language.

In [ ]:
%%sql -r create_semantic_view
CREATE OR REPLACE SEMANTIC VIEW TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.CALL_INSIGHTS_SV
  TABLES (
    seg AS TRANSCRIPT_CHURN_ML_DEMO.ML.SEGMENT_TOPICS PRIMARY KEY (SEGMENT_ID)
      COMMENT = 'One row per conversation segment with its topic',
    price AS TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.PRICE_SENTIMENT PRIMARY KEY (SEGMENT_ID)
      COMMENT = 'Price sentiment for segments that mention price',
    opp AS TRANSCRIPT_CHURN_ML_DEMO.RAW.CRM_OPPORTUNITIES PRIMARY KEY (OPPORTUNITY_ID)
      COMMENT = 'Renewal and upsell opportunities and their outcome',
    acct AS TRANSCRIPT_CHURN_ML_DEMO.RAW.ACCOUNTS PRIMARY KEY (ACCOUNT_ID)
      COMMENT = 'Customer accounts'
  )
  RELATIONSHIPS (
    price_to_segment AS price (SEGMENT_ID) REFERENCES seg,
    segment_to_opportunity AS seg (OPPORTUNITY_ID) REFERENCES opp,
    segment_to_account AS seg (ACCOUNT_ID) REFERENCES acct
  )
  DIMENSIONS (
    seg.topic AS seg.TOPIC_NAME COMMENT = 'Topic of the segment (business-defined or discovered)',
    seg.topic_source AS seg.TOPIC_SOURCE COMMENT = 'SEEDED, DISCOVERED or NOISE',
    seg.call_type AS seg.CALL_TYPE COMMENT = 'RENEWAL, SUPPORT, SALES or ONBOARDING',
    seg.call_month AS DATE_TRUNC('MONTH', seg.CALL_TS) COMMENT = 'Month of the call',
    price.price_driver AS price.PRICE_DRIVER COMMENT = 'Main driver of a negative price remark',
    price.price_sentiment AS price.OVERALL_SENTIMENT COMMENT = 'Overall sentiment of a price remark',
    opp.deal_outcome AS opp.OUTCOME COMMENT = 'WON or LOST',
    opp.opportunity_type AS opp.OPPORTUNITY_TYPE COMMENT = 'RENEWAL or UPSELL',
    acct.customer_segment AS acct.SEGMENT COMMENT = 'LOAN_OFFICER, AGENT or TEAM',
    acct.plan_tier AS acct.PLAN_TIER COMMENT = 'ESSENTIALS, PRO or ENTERPRISE'
  )
  METRICS (
    seg.segment_count AS COUNT(seg.SEGMENT_ID) COMMENT = 'Number of conversation segments',
    seg.call_count AS COUNT(DISTINCT seg.CALL_ID) COMMENT = 'Number of calls',
    price.price_remarks AS COUNT(price.SEGMENT_ID) COMMENT = 'Segments that mention price',
    price.negative_price_remarks AS SUM(IFF(price.IS_NEGATIVE, 1, 0)) COMMENT = 'Negative or mixed price remarks'
  )
  COMMENT = 'Call topics, price sentiment and deal outcomes'

In [ ]:
%%sql -r semantic_metrics
SHOW SEMANTIC METRICS IN TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.CALL_INSIGHTS_SV

In [ ]:
n_metrics = len(semantic_metrics)
n_dims = len(session.sql(f"SHOW SEMANTIC DIMENSIONS IN {DB}.ANALYTICS.CALL_INSIGHTS_SV").collect())
check("S3", n_metrics == 4 and n_dims == 10, f"ANALYTICS.CALL_INSIGHTS_SV: {n_dims} dimensions, {n_metrics} metrics")

---
## 4. Both questions, through the semantic view

The same two questions as Sections 1 and 2, asked in business terms. In Snowsight, open **AI & ML » Cortex Analyst**, pick
`CALL_INSIGHTS_SV` and ask *"What is driving negative price remarks?"* or *"Which topics come up on lost deals compared
with won deals?"*.

In [ ]:
%%sql -r sv_price_drivers
SELECT * FROM SEMANTIC_VIEW(TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.CALL_INSIGHTS_SV
  DIMENSIONS price.price_driver
  METRICS price.negative_price_remarks
  WHERE price.price_driver IS NOT NULL)
ORDER BY NEGATIVE_PRICE_REMARKS DESC

In [ ]:
%%sql -r sv_topics_by_outcome
SELECT * FROM SEMANTIC_VIEW(TRANSCRIPT_CHURN_ML_DEMO.ANALYTICS.CALL_INSIGHTS_SV
  DIMENSIONS opp.deal_outcome, seg.topic
  METRICS seg.segment_count)
ORDER BY DEAL_OUTCOME, SEGMENT_COUNT DESC

In [ ]:
sv_drivers = sv_price_drivers.set_index("PRICE_DRIVER").NEGATIVE_PRICE_REMARKS.astype(int).to_dict()
direct = {k: int(v) for k, v in price_negatives_by_driver.set_index("PRICE_DRIVER").NEGATIVE_SEGMENTS.to_dict().items()}
outcomes = {o for o in sv_topics_by_outcome.DEAL_OUTCOME if o}      # calls with no opportunity have no outcome
check("S4", sv_drivers == direct and {"WON", "LOST"} <= outcomes,
      f"semantic view drivers {sv_drivers} equal the direct SQL; topics by outcome cover {sorted(outcomes)} "
      f"({len(sv_topics_by_outcome)} rows)")

---
## Summary

Price complaints are found with AI on about a third of segments and attributed to a driver on the negatives only. Topics
line up with deal outcomes, and both answers sit behind one semantic view. Next: `04` turns call signals into churn
features.